## Countries

In [1]:
import pygadm
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from pathlib import Path
import pycountry
import os

## Rasters

In [2]:
checkpoint_path = "../results/country_confustion_matrix.parquet"
removed = ["AUS", "BRA", "CAN", "USA", "RUS", "GRL", "MEX", "CHL", "IDN"]

if os.path.exists(checkpoint_path):
    df = pd.read_parquet(checkpoint_path)
    completed = set(df.gid.to_list())
else:
    df = pd.DataFrame(columns=["country", "gid", "pixel_count", "TN", "FP", "FN", "TP"])
    completed = set()

completed.update(removed)

In [3]:
df

,country,gid,pixel_count,TN,FP,FN,TP
0,Zimbabwe,ZWE,1928357,1919765,7087,242,1263
1,Zambia,ZMB,3614517,3602594,8314,1630,1979
2,South Africa,ZAF,6513508,6326288,169600,51,17569
3,Yemen,YEM,2198652,2182165,24330,9228,10421
4,Samoa,WSM,13745,13431,1658,1409,1471
...,...,...,...,...,...,...,...
211,Albania,ALB,177532,165203,11643,0,686
212,Åland Islands,ALA,13830,13689,139,0,2
213,Angola,AGO,5977578,5948569,22542,979,5488
214,Afghanistan,AFG,3614259,3585597,24618,875,3169


In [4]:
error_path = "../results/country_errors.parquet"

if os.path.exists(error_path):
    error_df = pd.read_parquet(error_path)
    error = set(error_df.gid.to_list())
    completed.update(error)
else:
    error_df = pd.DataFrame(columns=["country", "gid", "error"])

In [5]:
error_df

,country,gid,error
0,Tuvalu,TUV,"('y', 'x') must be a permuted list of FrozenMa..."
1,Türkiye,TUR,Manifest from NASA DAAC (https://ladsweb.modap...
2,Svalbard and Jan Mayen,SJM,Manifest from NASA DAAC (https://ladsweb.modap...
3,Peru,PER,Manifest from NASA DAAC (https://ladsweb.modap...
4,Pakistan,PAK,Must have equal len keys and value when settin...
5,New Zealand,NZL,Only 1 dimenional array found. Cannot calculat...
6,Nauru,NRU,"('y', 'x') must be a permuted list of FrozenMa..."
7,Netherlands,NLD,Must have equal len keys and value when settin...
8,Namibia,NAM,Manifest from NASA DAAC (https://ladsweb.modap...
9,Marshall Islands,MHL,"('y', 'x') must be a permuted list of FrozenMa..."


In [ ]:
import shutil
import numpy as np
from conflict_monitoring_ntl.satellites import BlackMarblePy, GHSLSurface
from conflict_monitoring_ntl.transform import RasterPipeline
from conflict_monitoring_ntl.utils import binarize_xarray, get_combined_mask, get_non_nan_flat_array
from rasterio.enums import Resampling
import datetime
from sklearn.metrics import confusion_matrix

rasters = [GHSLSurface(), BlackMarblePy(frequency="monthly")]
transformations = [{"reproject_match": {"resampling": Resampling.sum}}, {}]
date = datetime.date(2020, 1, 1)

with tqdm(pycountry.countries, desc="Calculating confusion matrix:") as pbar:

    for country_dto in pbar:
        
        country = country_dto.name
        gid = country_dto.alpha_3

        pbar.set_postfix(country=country)

        if gid in completed:
            continue

        try:
            
            gdf = pygadm.Items(admin=gid, content_level=1)
        
            pixels = 0
            conf_mat = np.zeros(4, dtype=np.int64)


            for i in tqdm(range(len(gdf)), desc="Processing Provinces"):

                province_gdf = gpd.GeoDataFrame(gdf.iloc[[i]].geometry).set_crs("EPSG:4326")

                pipeline = RasterPipeline(province_gdf, date, rasters, transformations)
                ds = pipeline.run()

                # make sure we compare non-nan areas
                mask = get_combined_mask(ds)
                ds = ds.where(mask)

                ghsl_pop_binary = binarize_xarray(ds.ghsl_surface, 2500)
                y_true = get_non_nan_flat_array(ghsl_pop_binary)

                bm_binary = binarize_xarray(ds.black_marble_radiance_monthly, 1.0)
                y_pred = get_non_nan_flat_array(bm_binary)

                assert y_pred.shape == y_true.shape

                conf_mat += confusion_matrix(y_true, y_pred).flatten()
                pixels += mask.sum().item()

            data = [country, gid, pixels, *conf_mat.tolist()]
            df = pd.concat([pd.DataFrame([data], columns=df.columns), df], ignore_index=True)
            df.to_parquet(checkpoint_path)

            bm_path = Path(os.path.abspath('')).parent / "data" / "black_marble"
            shutil.rmtree(bm_path)  
            bm_path.mkdir(exist_ok=True)
                
        except Exception as e:

            data = [country, gid, str(e)]
            error_df = pd.concat([pd.DataFrame([data], columns=error_df.columns), error_df], ignore_index=True)
            error_df.to_parquet(error_path)